# Validation des données COMPLETUDE pour le TdB DSE RDC  
Esteban Montandon
05 Juin 2024

In [ ]:
install.packages('stringdist', quiet = TRUE)

suppressPackageStartupMessages({
    library(stringdist)
    library(readxl)
    library(data.table)
    library(sf)
    library(DBI)
    library(stringr)
    })

In [ ]:
# parameters
# nouvelles_completude_path = "/home/hexa/workspace/tdb-suivi-epidemio/TEST_CHECKS/COMPLETUDE PROMPTITUDE SURVEILLANCE_Week20.xlsx"
# nouvelles_completude_path = "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/Completude/2025/COMPLETUDE PROMPTITUDE SURVEILLANCE_Week14.xlsx"
# nouvelles_completude_path = "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/Completude/2026/COMPLETUDE PROMPTITUDE SURVEILLANCE_Week05.xlsx"

In [ ]:
# Read the file
sheet.names <- as.list(excel_sheets(nouvelles_completude_path))

# remove sheets that are improperly named
sheet.names <- sheet.names[!(sheet.names %like% "Feuil")]

# Print existing tabs
cat("Document tabs: [", paste(sheet.names, collapse = ", "), "]\n", sep = "")

##### 1) Check all tabs are present in the excel file  

In [ ]:
# extract tab numbers
sheet.nrs <- as.numeric(str_extract(sheet.names, "\\d+"))
cat("Tab numbers:" , paste(sheet.nrs, collapse = ", "))

In [ ]:
# Extract file name
file.name <- basename(nouvelles_completude_path)

# Get Week number from file name
pattern <- "(?i)(?:^|_|\\s|-)week(\\d+)\\s*" 
match <- str_extract(file.name, pattern)
week.nr <- as.numeric(str_extract(match, "\\d+"))
# cat("Expected tab numbers in" , file.name, ":", paste(1:week.nr, collapse = ", "))


In [ ]:
# Check if expected tabs are there 
if (!all(1:week.nr %in% sheet.nrs)) {
    missing.tabs <- setdiff(1:week.nr, sheet.nrs)
    msg <- paste0("\nError: Veuillez vérifier que le fichier: ", file.name, 
                  " contient tous les onglets prévus. Onglets non trouvés: ", paste("SE", missing.tabs, collapse = ", ", sep=""), "\n") 
    stop(msg)
}

##### 2) Check all tabs match the title

In [ ]:
# Read the sheet by name
named_list <- setNames(as.list(sheet.nrs), sheet.names)

wrong_title_sheets <- list()
for (sname in names(named_list)) {
    # Read the first cell (A1) of the current sheet to get the title nr
    suppressWarnings({
        title_cell <- read_excel(nouvelles_completude_path, sheet = sname, range = "A1", col_names = TRUE)        
        nr_match <- as.numeric(trimws(str_extract(names(title_cell)[1], "\\s(\\d+)\\s*")))
        })   
    if (named_list[[sname]] != nr_match) {
        wrong_title_sheets[[sname]] <- nr_match
    }
}

In [ ]:
if (length(wrong_title_sheets) > 0) {    
    # Convert named list to string format with line breaks
    result_string <- sapply(seq_along(wrong_title_sheets), function(i) {
      paste("Le numéro du titre sur l'onglet: ", names(wrong_title_sheets)[i], " ne correspond pas (", wrong_title_sheets[[i]], ")")
    })
    stop(paste0("\nError: ", paste(result_string, collapse = ", "), "\n"))
}